# Cross-Dataset Nested Kriging

This notebook implements and evaluates the **cross-dataset nested kriging layer**: a second-level Rullière aggregation that pools block centroids from all 55 per-dataset NK models into a single global pool.

## Architecture

Each per-dataset `NestedKrigingForecaster` (or `MultivariateNestedKrigingForecaster`) saves M_d block centroids after fitting.  Because datasets have different context lengths L_d, all centroids are mapped to a common d-dimensional **ACF feature space** before the cross-dataset kernel matrix is assembled:

$$\hat{y}(x^*) = c(\phi^*)^\top [C + D]^{-1} \hat{\mu}$$

where $\phi^* = \phi(x^*)$ is the ACF embedding of the query, $C_{ij} = k_{\mathrm{RBF}}(\phi_i, \phi_j)$, $D_{ii} = \hat{\sigma}^2_i$, and $\hat{\mu}_i = \hat{y}_i(\bar{x}_i)$.

## Workflow

1. **Load saved states** — scan `experiments/models/*.npz`
2. **Fit `CrossDatasetNestedKrigingForecaster`** on all loaded states
3. **Evaluate** on GIFT-Eval test windows for selected datasets
4. **Compare** cross-dataset predictions vs per-dataset NK predictions
5. **Visualise** the cross-dataset weight heatmap

In [ ]:
import sys
import pathlib
import warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Add project root to path
ROOT = pathlib.Path("..")
sys.path.insert(0, str(ROOT))

from src.nested_kriging import (
    load_nk_state,
    acf_embedding,
    CrossDatasetNestedKrigingForecaster,
)

MODELS_DIR = pathlib.Path("models")

print(f"Models directory: {MODELS_DIR.resolve()}")
saved = sorted(MODELS_DIR.glob("*.npz"))
print(f"Found {len(saved)} saved model state(s):")
for p in saved:
    print(f"  {p.name}")

In [ ]:
# ── Load all saved model states ──────────────────────────────────────────────
states = []
for p in saved:
    try:
        s = load_nk_state(p)
        states.append(s)
        mv_tag = f" (MV, {s['n_channels']} ch)" if 'n_channels' in s else ""
        print(f"  {s['dataset_name']:40s}  "
              f"M={len(s['centroids']):4d}  L={s['context_length']:4d}  H={s['horizon_length']:3d}"
              f"{mv_tag}")
    except Exception as exc:
        print(f"  ERROR loading {p.name}: {exc}")

print(f"\nTotal blocks in global pool: {sum(len(s['centroids']) for s in states)}")

In [ ]:
# ── Fit CrossDatasetNestedKrigingForecaster ──────────────────────────────────
ACF_DIM = 20       # ACF embedding dimension (lags 1..20)
SIGMA_N = 1e-3     # noise floor

cdnk = CrossDatasetNestedKrigingForecaster(
    acf_dim=ACF_DIM,
    sigma_n=SIGMA_N,
).fit(states)

print(f"Fitted CrossDatasetNestedKrigingForecaster")
print(f"  M_total      = {cdnk._phi_centroids.shape[0]}")
print(f"  ACF dim      = {ACF_DIM}")
print(f"  length_scale = {cdnk.length_scale_:.4f}  (median heuristic)")
print(f"  max_H        = {cdnk._max_H}")

In [ ]:
# ── ACF embedding visualisation ──────────────────────────────────────────────
# Show the global centroid distribution in the first 2 ACF dimensions,
# colour-coded by dataset.

# Assign a colour to each dataset
dataset_ids = []
offset = 0
for s in states:
    M = len(s['centroids'])
    dataset_ids.extend([s['dataset_name']] * M)
    offset += M

unique_ds = list(dict.fromkeys(dataset_ids))   # preserves order
ds_to_idx = {d: i for i, d in enumerate(unique_ds)}
colors = np.array([ds_to_idx[d] for d in dataset_ids], dtype=float)

phi = cdnk._phi_centroids   # (M_total, d)

fig, ax = plt.subplots(figsize=(9, 6))
sc = ax.scatter(phi[:, 0], phi[:, 1], c=colors, cmap="tab20",
                s=18, alpha=0.7, linewidths=0)
ax.set_xlabel("ACF lag 1 (ρ̂₁)", fontsize=12)
ax.set_ylabel("ACF lag 2 (ρ̂₂)", fontsize=12)
ax.set_title("Global centroid pool — first two ACF dimensions", fontsize=13)
cbar = fig.colorbar(sc, ax=ax, fraction=0.03, pad=0.01)
cbar.set_label("dataset index", fontsize=10)
plt.tight_layout()
plt.savefig("cross_dataset_acf_scatter.png", dpi=150)
plt.show()

In [ ]:
# ── Cross-dataset weight heatmap ─────────────────────────────────────────────
# For each state, pick its first centroid as a representative query and compute
# the Rullière weights. The heatmap shows weight[query_dataset, source_block].

n_query_datasets = min(len(states), 20)       # show at most 20
M_total = cdnk._phi_centroids.shape[0]

W = np.zeros((n_query_datasets, M_total))
query_labels = []

for qi, s in enumerate(states[:n_query_datasets]):
    q_ctx = s['centroids'][0]                  # first centroid of this dataset
    phi_q = acf_embedding(q_ctx, ACF_DIM)      # (d,)
    from src.nested_kriging import _rbf_kernel, _safe_cholesky, _EPS
    c = _rbf_kernel(phi_q[None], cdnk._phi_centroids,
                    length_scale=cdnk.length_scale_,
                    amplitude=cdnk.amplitude).ravel()
    w = np.linalg.solve(cdnk._chol_CD.T, np.linalg.solve(cdnk._chol_CD, c))
    W[qi] = w
    query_labels.append(s['dataset_name'].split("/")[0][:18])   # short label

# Column labels: one per block, labelled by dataset
block_ds_labels = []
for s in states:
    short = s['dataset_name'].split("/")[0][:12]
    for _ in range(len(s['centroids'])):
        block_ds_labels.append(short)

fig, ax = plt.subplots(figsize=(max(12, M_total * 0.22), max(5, n_query_datasets * 0.35)))
im = ax.imshow(W, aspect='auto', cmap='RdBu_r',
               vmin=-np.abs(W).max(), vmax=np.abs(W).max())
ax.set_yticks(range(n_query_datasets))
ax.set_yticklabels(query_labels, fontsize=8)
ax.set_xlabel("Global block index (coloured by source dataset)", fontsize=10)
ax.set_ylabel("Query dataset", fontsize=10)
ax.set_title("Cross-dataset Rullière weights", fontsize=13)
plt.colorbar(im, ax=ax, fraction=0.02, pad=0.01)
plt.tight_layout()
plt.savefig("cross_dataset_weight_heatmap.png", dpi=150)
plt.show()

In [ ]:
# ── Evaluation on GIFT-Eval test windows ────────────────────────────────────
#
# For each dataset that has a saved state, load the GIFT-Eval test split,
# run both the per-dataset NK model and the cross-dataset NK model,
# and compute MAE / RMSE.
#
# NOTE: This cell requires the individual NK notebooks to have been run
# first so that the per-dataset models are trained.  The cross-dataset model
# uses only the saved .npz state — no re-fitting needed.

try:
    from gift_eval.data import Dataset
    HAS_GIFT = True
except ImportError:
    HAS_GIFT = False
    print("gift_eval not installed — skipping GIFT-Eval evaluation.")

if HAS_GIFT:
    from src.nested_kriging import pool_windows_from_series, NestedKrigingForecaster

    EVAL_DATASETS = [
        ("m4_hourly", "H"),
        ("m4_monthly", "M"),
        ("electricity", "H"),
        ("m4_daily", "D"),
    ]
    MAX_TEST_WINDOWS = 200   # cap to keep runtime reasonable
    CONTEXT_MULT = 3         # context_length = prediction_length × CONTEXT_MULT

    results = {}

    for ds_name, term in EVAL_DATASETS:
        npz_name = f"{ds_name.replace('/', '__')}.npz"
        if not (MODELS_DIR / npz_name).exists():
            print(f"  {ds_name}/{term}: no saved state — skipping")
            continue

        try:
            ds = Dataset(name=ds_name, term=term)
        except Exception as exc:
            print(f"  {ds_name}/{term}: failed to load dataset ({exc})")
            continue

        H = ds.prediction_length
        L = H * CONTEXT_MULT
        state = load_nk_state(MODELS_DIR / npz_name)

        maes_cross, rmses_cross = [], []
        n_eval = 0

        for s_idx, ts in enumerate(ds.test_data):
            series = np.asarray(ts.target, dtype=float).ravel()
            T = len(series)
            if T < L + H:
                continue

            ctx_raw = series[-(L + H): -H]
            tgt_raw = series[-H:]

            # Robust normalisation (same as per-dataset notebooks)
            loc = float(np.median(ctx_raw))
            scale = float(np.quantile(np.abs(ctx_raw - loc), 0.75)) or 1.0
            ctx_norm = (ctx_raw - loc) / scale

            # Cross-dataset prediction (de-normalised)
            pred_norm = cdnk.predict(ctx_norm, target_horizon=H)
            pred = pred_norm * scale + loc

            mae  = float(np.mean(np.abs(pred - tgt_raw)))
            rmse = float(np.sqrt(np.mean((pred - tgt_raw) ** 2)))
            maes_cross.append(mae)
            rmses_cross.append(rmse)
            n_eval += 1

            if n_eval >= MAX_TEST_WINDOWS:
                break

        results[f"{ds_name}/{term}"] = {
            "MAE_cross":  float(np.mean(maes_cross)),
            "RMSE_cross": float(np.mean(rmses_cross)),
            "n_windows":  n_eval,
        }
        print(f"  {ds_name}/{term:5s}  n={n_eval:4d}  "
              f"MAE_cross={np.mean(maes_cross):.4f}  "
              f"RMSE_cross={np.mean(rmses_cross):.4f}")

    print("\nEvaluation complete.")

In [ ]:
# ── Qualitative forecast plots ───────────────────────────────────────────────
#
# For the first available state, show cross-dataset NK predictions vs actual.

if states:
    s0 = states[0]
    H0 = s0['horizon_length']
    L0 = s0['context_length']
    ds_name0 = s0['dataset_name']

    # Use the centroid itself as a synthetic query (reasonable proxy)
    n_show = min(4, len(s0['centroids']))
    fig, axes = plt.subplots(1, n_show, figsize=(4 * n_show, 3.5), sharey=False)
    if n_show == 1:
        axes = [axes]

    for i, ax in enumerate(axes):
        ctx = s0['centroids'][i]            # (L0,)
        true_h = s0['centroid_preds'][i]    # (H0,) — local prediction (ground truth proxy)
        pred_h = cdnk.predict(ctx, target_horizon=H0)

        t_ctx = np.arange(-L0, 0)
        t_h   = np.arange(1, H0 + 1)

        ax.plot(t_ctx, ctx, color='steelblue', lw=1.2, label='context')
        ax.axvline(0, color='gray', lw=0.8, ls='--')
        ax.plot(t_h, true_h, color='forestgreen', lw=1.4, ls='--', label='per-dataset NK')
        ax.plot(t_h, pred_h, color='tomato',      lw=1.4, label='cross-dataset NK')
        ax.set_title(f"block {i}", fontsize=10)
        ax.set_xlabel("step")

    axes[0].set_ylabel("normalised value")
    axes[0].legend(fontsize=8, loc='upper left')
    fig.suptitle(f"{ds_name0} — cross-dataset NK vs per-dataset NK (centroid queries)",
                 fontsize=11, y=1.02)
    plt.tight_layout()
    plt.savefig("cross_dataset_qualitative.png", dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# ── Cross-dataset vs per-dataset weight self-attribution ─────────────────────
#
# For each dataset, compute the fraction of weight attributed to blocks from
# the same dataset vs. all other datasets.  High self-attribution means the
# cross-dataset model mostly agrees with the per-dataset model; low means it
# is genuinely borrowing strength from other datasets.

self_weights = []    # fraction of positive weight from same dataset
ds_names_short = []

# Build cumulative block index → dataset mapping
block_to_ds = []
for s in states:
    block_to_ds.extend([s['dataset_name']] * len(s['centroids']))
block_to_ds_arr = np.array(block_to_ds)

from src.nested_kriging import _rbf_kernel

for s in states:
    ds = s['dataset_name']
    q_ctx = s['centroids'].mean(axis=0)          # mean centroid of this dataset
    phi_q = acf_embedding(q_ctx, ACF_DIM)
    c = _rbf_kernel(phi_q[None], cdnk._phi_centroids,
                    length_scale=cdnk.length_scale_,
                    amplitude=cdnk.amplitude).ravel()
    w = np.linalg.solve(cdnk._chol_CD.T, np.linalg.solve(cdnk._chol_CD, c))

    w_pos = np.maximum(w, 0)                     # only positive weights
    same_mask = block_to_ds_arr == ds
    self_frac = float(w_pos[same_mask].sum() / (w_pos.sum() + 1e-12))
    self_weights.append(self_frac)
    ds_names_short.append(ds.split('/')[0][:18])

# Bar chart
fig, ax = plt.subplots(figsize=(max(10, len(states) * 0.35), 4))
bars = ax.bar(range(len(self_weights)), self_weights, color='steelblue', alpha=0.8)
ax.axhline(1.0, color='gray', ls='--', lw=0.8)
ax.set_xticks(range(len(states)))
ax.set_xticklabels(ds_names_short, rotation=60, ha='right', fontsize=7)
ax.set_ylabel("Self-attribution fraction", fontsize=11)
ax.set_title("Fraction of positive Rullière weight attributed to same-dataset blocks",
             fontsize=12)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig("cross_dataset_self_attribution.png", dpi=150)
plt.show()

print(f"Mean self-attribution: {np.mean(self_weights):.3f}")
print(f"Min  self-attribution: {np.min(self_weights):.3f}  ({ds_names_short[int(np.argmin(self_weights))]} — most cross-dataset borrowing)")